# FLIDS Day 3 Extension — Chronological Temporal Holdout & Feature Ablation

Closes the two remaining top-tier journal reviewer gaps in a single Kaggle CPU run (~1.2–1.5 hours, zero GPU queue):
1. **Part 1 — Strict Chronological Holdout (`NF-UNSW-NB15-v3-temporal`, 100,276 flows):**
   Blocks 0–1 (Fit) → Block 2 (Early Stopping) → Block 3 (Source Calibration) → Block 4 (Future Target Test) across 8 methods × 5 seeds = 40 runs.
2. **Part 2 — Matched Retuned Feature Ablation (`no_ports` 39 features & `no_ports_ttl` 37 features):**
   5 core methods (`LogisticRegression`, `LightGBM`, `FedLogReg`, `FedAvg`, `FedNova`) × 2 variants × 4 datasets × 5 seeds = 200 runs.

### Kaggle Setup
1. Accelerator: **None (CPU)** (fast, zero queue!) or **GPU P100**.
2. Input: attach **`fragoa/notebook6ceac598dd`** (contains both `temporal_cache.npz` and `nf_cache.npz` inside `FLIDS_Master_V3_Fresh_Results.zip`).
3. Click **Save Version → Save & Run All (Commit)**.

In [ ]:
# Configuration
SMOKE_TEST = False
RUN_TEMPORAL = True
RUN_ABLATION = True
WORK_DIR = '/kaggle/working/day3_work'


In [ ]:
# Write embedded day3_temporal_ablation.py module
from pathlib import Path
CODE = '"""FLIDS Day 3 Extension: Chronological Temporal Holdout (NF-UNSW-NB15-v3) + Matched Feature Ablation.\n\nPart 1 — Strict Chronological Holdout (`temporal_cache.npz`, 100,276 flows, 5 time blocks):\n    - Fit: Blocks 0-1 (3 Dirichlet alpha=0.5 FL clients)\n    - Early Stopping / Model Selection: Block 2\n    - Source Calibration: Block 3\n    - Future Target Test: Block 4 (strictly forward in time, zero overlap)\n    - Methods (8 x 5 seeds = 40 runs):\n      LogisticRegression, RandomForest, LightGBM, FedAvg, FedNova, SCAFFOLD, FedLogReg, FedBN_TTA\n\nPart 2 — Matched, Retuned NetFlow Feature Ablation (`nf_cache.npz`, 93,810 flows):\n    - Feature variants:\n      * no_ports (39 features: drops L4_SRC_PORT, L4_DST_PORT)\n      * no_ports_ttl (37 features: drops L4_SRC_PORT, L4_DST_PORT, MIN_TTL, MAX_TTL)\n    - Exact same split digests, 4-candidate inner LOSO selection, and 150-round training protocol\n    - Methods (5 core x 2 variants x 4 domains x 5 seeds = 200 runs):\n      LogisticRegression, LightGBM, FedLogReg, FedAvg, FedNova\n\nUsage:\n    python experiments/day3/day3_temporal_ablation.py --input /path/to/FLIDS_Master_V3_Fresh_Results.zip --work ./day3_work\n"""\nimport os\nfor _k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):\n    os.environ[_k] = "1"\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport math\nimport time\nimport zipfile\nfrom pathlib import Path\n\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom lightgbm import LGBMClassifier\nfrom scipy.stats import binom\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom torch import nn\n\nSEEDS = [0, 42, 3407, 123, 2026]\nDOMAINS = ["NF-UNSW-NB15", "NF-ToN-IoT", "NF-BoT-IoT", "NF-CSE-CIC-IDS2018"]\nFEATURES = [\n    "L4_SRC_PORT", "L4_DST_PORT", "PROTOCOL", "L7_PROTO",\n    "IN_BYTES", "OUT_BYTES", "IN_PKTS", "OUT_PKTS",\n    "FLOW_DURATION_MILLISECONDS", "TCP_FLAGS", "CLIENT_TCP_FLAGS", "SERVER_TCP_FLAGS",\n    "DURATION_IN", "DURATION_OUT", "MIN_TTL", "MAX_TTL",\n    "LONGEST_FLOW_PKT", "SHORTEST_FLOW_PKT", "MIN_IP_PKT_LEN", "MAX_IP_PKT_LEN",\n    "SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES",\n    "RETRANSMITTED_IN_BYTES", "RETRANSMITTED_IN_PKTS",\n    "RETRANSMITTED_OUT_BYTES", "RETRANSMITTED_OUT_PKTS",\n    "SRC_TO_DST_AVG_THROUGHPUT", "DST_TO_SRC_AVG_THROUGHPUT",\n    "NUM_PKTS_UP_TO_128_BYTES", "NUM_PKTS_128_TO_256_BYTES",\n    "NUM_PKTS_256_TO_512_BYTES", "NUM_PKTS_512_TO_1024_BYTES",\n    "NUM_PKTS_1024_TO_1514_BYTES", "TCP_WIN_MAX_IN", "TCP_WIN_MAX_OUT",\n    "ICMP_TYPE", "ICMP_IPV4_TYPE", "DNS_QUERY_ID", "DNS_QUERY_TYPE",\n    "DNS_TTL_ANSWER", "FTP_COMMAND_RET_CODE",\n]\n\nTEMPORAL_METHODS = [\n    "LogisticRegression", "RandomForest", "LightGBM",\n    "FedAvg", "FedNova", "SCAFFOLD", "FedLogReg", "FedBN_TTA",\n]\nABLATION_METHODS = ["LogisticRegression", "LightGBM", "FedLogReg", "FedAvg", "FedNova"]\nABLATION_VARIANTS = {\n    "no_ports": ["L4_SRC_PORT", "L4_DST_PORT"],\n    "no_ports_ttl": ["L4_SRC_PORT", "L4_DST_PORT", "MIN_TTL", "MAX_TTL"],\n}\n\n\ndef make_seed(*args) -> int:\n    return int.from_bytes(hashlib.sha256(repr(args).encode()).digest()[:4], "little")\n\n\ndef make_rng(*args) -> np.random.Generator:\n    return np.random.default_rng(make_seed(*args))\n\n\ndef signed_log(x: np.ndarray) -> np.ndarray:\n    c = np.clip(np.asarray(x, dtype=np.float64), -1e12, 1e12)\n    return np.clip(np.sign(c) * np.log1p(np.abs(c)), -18.0, 18.0).astype(np.float32)\n\n\ndef save_json(path: Path, obj) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef npwrite(path: Path, **kw) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(".tmp")\n    with tmp.open("wb") as f:\n        np.savez_compressed(f, **kw)\n    tmp.replace(path)\n\n\ndef emp_cut(scores: np.ndarray, alpha: float) -> float:\n    s = np.sort(np.asarray(scores, dtype=np.float64))\n    if not len(s):\n        return float("inf")\n    k = math.floor(alpha * len(s))\n    return float(np.nextafter(s[len(s) - k - 1], np.inf))\n\n\ndef wilks_cut(scores: np.ndarray, alpha: float, delta: float = 0.05) -> float:\n    s = np.sort(np.asarray(scores, dtype=np.float64))\n    n = len(s)\n    k = int(binom.ppf(1.0 - delta, n, 1.0 - alpha)) + 1\n    return float(np.nextafter(s[k - 1], np.inf)) if k <= n else float("inf")\n\n\ndef pauc(y: np.ndarray, s: np.ndarray, max_fpr=None) -> float:\n    return float(roc_auc_score(y, s, max_fpr=max_fpr))\n\n\ndef eval_op(y: np.ndarray, s: np.ndarray, thr: float):\n    n = s[y == 0]\n    p = s[y == 1]\n    fp = int((n >= thr).sum())\n    tp = int((p >= thr).sum())\n    return fp / len(n), tp / len(p), fp, tp\n\n\ndef groupsplit(ids, y, s, frac):\n    a, b = train_test_split(np.asarray(ids), test_size=frac, random_state=s, stratify=y[ids])\n    return np.sort(a), np.sort(b)\n\n\ndef split_outer_nf(y, d, held, s):\n    src = np.flatnonzero(~np.isin(d, held))\n    test = np.flatnonzero(np.isin(d, held))\n    domains = sorted(set(d[src]))\n    fit, stop, cal, st = [], [], [], []\n    for dom in domains:\n        ids = src[d[src] == dom]\n        a, b = groupsplit(ids, y, make_seed(s, dom, "outer"), 0.30)\n        c, e = groupsplit(b, y, make_seed(s, dom, "cal"), 2 / 3)\n        g, h = groupsplit(e, y, make_seed(s, dom, "st"), 0.5)\n        fit.extend(a); stop.extend(c); cal.extend(g); st.extend(h)\n    return tuple(np.array(v, dtype=int) for v in (fit, stop, cal, st, test))\n\n\ndef dirichlet_clients(ids, y, s, K=3, alpha=0.5):\n    r = make_rng(s, "dirichlet_temporal")\n    buckets = [[] for _ in range(K)]\n    for lab in (0, 1):\n        a = r.permutation(ids[y[ids] == lab])\n        p = r.dirichlet(np.full(K, alpha))\n        chunks = np.split(a, (np.cumsum(p)[:-1] * len(a)).astype(int))\n        for j, c in enumerate(chunks):\n            buckets[j].extend(c)\n    return [np.asarray(b, dtype=int) for b in buckets if len(b) >= 4]\n\n\ndef candidate_grid(method: str):\n    if method == "LogisticRegression":\n        return [dict(C=v) for v in (0.01, 0.1, 1.0, 10.0)]\n    if method == "RandomForest":\n        return [dict(max_depth=d, min_samples_leaf=l) for d, l in [(12, 1), (None, 1), (16, 5), (None, 5)]]\n    if method == "LightGBM":\n        return [dict(learning_rate=l, num_leaves=n) for l, n in [(0.03, 31), (0.1, 31), (0.03, 63), (0.1, 63)]]\n    if method == "FedLogReg":\n        return [dict(lr=0.01, wd=1e-2), dict(lr=0.01, wd=1e-3), dict(lr=0.03, wd=1e-3), dict(lr=0.03, wd=1e-4)]\n    if method == "SCAFFOLD":\n        return [dict(lr=l) for l in (0.001, 0.003, 0.01, 0.03)]\n    return [dict(lr=l) for l in (0.0003, 0.001, 0.003, 0.01)]\n\n\nclass LinearLogReg(nn.Module):\n    def __init__(self, n_in: int):\n        super().__init__()\n        self.net = nn.Linear(n_in, 1)\n\n    def forward(self, x):\n        return self.net(x).squeeze(-1)\n\n\nclass MLP(nn.Module):\n    def __init__(self, n_in: int):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))\n\n    def forward(self, x):\n        return self.net(x).squeeze(-1)\n\n\nclass MLPBN(nn.Module):\n    def __init__(self, n_in: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(n_in, 64), nn.BatchNorm1d(64), nn.ReLU(),\n            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(),\n            nn.Linear(32, 1),\n        )\n\n    def forward(self, x):\n        return self.net(x).squeeze(-1)\n\n\ndef score_model(net: nn.Module, xx: torch.Tensor, ids: np.ndarray) -> np.ndarray:\n    net.eval()\n    out = []\n    with torch.no_grad():\n        for chunk in np.array_split(ids, max(1, math.ceil(len(ids) / 4096))):\n            out.append(torch.sigmoid(net(xx[chunk])).cpu().numpy())\n    return np.concatenate(out).astype(np.float64)\n\n\ndef adapt_bn_unsupervised(nets, xx: torch.Tensor, unlabeled_ids: np.ndarray, batch_size: int = 512):\n    adapted = []\n    for src_net in nets:\n        net = MLPBN(xx.shape[1]).to(xx.device)\n        net.load_state_dict({k: v.detach().clone() for k, v in src_net.state_dict().items()})\n        for m in net.modules():\n            if isinstance(m, nn.BatchNorm1d):\n                m.reset_running_stats()\n                m.momentum = None\n        net.train()\n        with torch.no_grad():\n            for chunk in np.array_split(unlabeled_ids, max(1, math.ceil(len(unlabeled_ids) / batch_size))):\n                if len(chunk) >= 2:\n                    _ = net(xx[chunk])\n        net.eval()\n        adapted.append(net)\n    return adapted\n\n\ndef fit_method(raw, y, d, tr, va, method, cfg, s, opts, clients_override=None):\n    t0 = time.perf_counter()\n    trans = signed_log(raw)\n    sc = StandardScaler().fit(trans[tr])\n    X = np.clip(sc.transform(trans).astype(np.float32), -20.0, 20.0)\n\n    if method in ("LogisticRegression", "RandomForest", "LightGBM"):\n        if method == "LogisticRegression":\n            m = LogisticRegression(C=cfg["C"], max_iter=2000, random_state=s)\n            m.fit(X[tr], y[tr])\n        elif method == "RandomForest":\n            m = RandomForestClassifier(n_estimators=10 if opts["smoke"] else 300, n_jobs=2, random_state=s, **cfg)\n            m.fit(X[tr], y[tr])\n        else:\n            m = LGBMClassifier(n_estimators=15 if opts["smoke"] else 1500, verbosity=-1, n_jobs=2, random_state=s, **cfg)\n            m.fit(\n                X[tr], y[tr],\n                eval_set=[(X[va], y[va])],\n                eval_metric="binary_logloss",\n                callbacks=[lgb.early_stopping(50, verbose=False)],\n            )\n        return (lambda ids: m.predict_proba(X[ids])[:, 1].astype(np.float64)), None, dict(seconds=time.perf_counter() - t0)\n\n    device = opts["device"]\n    xx = torch.as_tensor(X, device=device)\n    yy = torch.as_tensor(y, dtype=torch.float32, device=device)\n    torch.manual_seed(s)\n    r = make_rng(s, method)\n\n    clients = clients_override if clients_override is not None else [tr[d[tr] == dom] for dom in sorted(set(d[tr]))]\n    rounds = opts["rounds"]\n    local_steps = opts["local_steps"]\n    check = opts["check_every"]\n    patience = opts["patience"]\n    min_round = opts["min_round"]\n    B = opts["batch"]\n\n    best = -np.inf\n    best_state = None\n    wait = 0\n\n    if method == "FedBN_TTA":\n        nets = [MLPBN(X.shape[1]).to(device) for _ in clients]\n        init = nets[0].state_dict()\n        for n in nets:\n            n.load_state_dict(init)\n        shared = [k for k in init if k.startswith(("net.0.", "net.3.", "net.6."))]\n        state = {k: init[k].detach().clone() for k in shared}\n\n        def sync():\n            for n in nets:\n                st = n.state_dict(); st.update(state); n.load_state_dict(st)\n\n        def pred_src(ids):\n            return np.mean([score_model(n, xx, ids) for n in nets], axis=0)\n\n        for rd in range(1, rounds + 1):\n            sync()\n            w = np.array([len(c) for c in clients], dtype=float); w /= w.sum()\n            updates = []\n            for i, ids in enumerate(clients):\n                n = nets[i]; n.train()\n                opt = torch.optim.Adam(n.parameters(), lr=cfg["lr"])\n                for _ in range(local_steps):\n                    b = r.choice(ids, max(2, min(B, len(ids))), replace=len(ids) < 2)\n                    opt.zero_grad()\n                    loss = nn.functional.binary_cross_entropy_with_logits(n(xx[b]), yy[b])\n                    loss.backward(); opt.step()\n                updates.append({k: n.state_dict()[k].detach().clone() for k in shared})\n            state = {k: sum(float(a) * u[k] for a, u in zip(w, updates)) for k in shared}\n            sync()\n            if rd % check == 0 or rd == rounds:\n                v = pauc(y[va], pred_src(va), 0.05)\n                if v > best + 1e-5:\n                    best = v\n                    best_state = [{k: val.detach().cpu().clone() for k, val in n.state_dict().items()} for n in nets]\n                    wait = 0\n                else:\n                    wait += 1\n            if rd >= min_round and wait >= patience:\n                break\n        for n, st in zip(nets, best_state):\n            n.load_state_dict(st)\n\n        def pred_tta(ids, adapt_ids=None):\n            if adapt_ids is None:\n                return pred_src(ids)\n            adapted = adapt_bn_unsupervised(nets, xx, adapt_ids)\n            return np.mean([score_model(n, xx, ids) for n in adapted], axis=0)\n\n        return pred_src, pred_tta, dict(seconds=time.perf_counter() - t0)\n\n    def build_net():\n        return LinearLogReg(X.shape[1]).to(device) if method == "FedLogReg" else MLP(X.shape[1]).to(device)\n\n    model = build_net()\n    mean_client_len = float(np.mean([len(c) for c in clients]))\n    cglobal = [torch.zeros_like(p) for p in model.parameters()]\n    clocal = [[torch.zeros_like(p) for p in model.parameters()] for _ in clients]\n\n    for rd in range(1, rounds + 1):\n        state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        weights = np.array([len(c) for c in clients], dtype=float); weights /= weights.sum()\n        deltas, counts, dc = [], [], []\n        for i, ids in enumerate(clients):\n            local = build_net(); local.load_state_dict(state); local.train()\n            if method in ("FedNova", "SCAFFOLD"):\n                opt = torch.optim.SGD(local.parameters(), lr=cfg["lr"])\n            elif method == "FedLogReg":\n                opt = torch.optim.Adam(local.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])\n            else:\n                opt = torch.optim.Adam(local.parameters(), lr=cfg["lr"])\n            nsteps = local_steps if method != "FedNova" else max(1, min(4 * local_steps, round(local_steps * len(ids) / mean_client_len)))\n            ref = [p.detach().clone() for p in local.parameters()]\n            control = [a - b for a, b in zip(cglobal, clocal[i])] if method == "SCAFFOLD" else None\n            for _ in range(nsteps):\n                b = r.choice(ids, min(B, len(ids)), replace=False)\n                opt.zero_grad()\n                loss = nn.functional.binary_cross_entropy_with_logits(local(xx[b]), yy[b])\n                loss.backward()\n                if control is not None:\n                    for p, c_vec in zip(local.parameters(), control):\n                        p.grad.add_(c_vec)\n                opt.step()\n            deltas.append({k: v.detach() - state[k] for k, v in local.state_dict().items()})\n            counts.append(nsteps)\n            if method == "SCAFFOLD":\n                new_c = [old - g + (q - p.detach()) / (nsteps * cfg["lr"]) for old, g, q, p in zip(clocal[i], cglobal, ref, local.parameters())]\n                dc.append([a - b for a, b in zip(new_c, clocal[i])])\n                clocal[i] = new_c\n        if method == "SCAFFOLD":\n            for j, g in enumerate(cglobal):\n                g.add_(sum(v[j] for v in dc) / len(clients))\n        if method == "FedNova":\n            tau_eff = float(np.dot(weights, counts))\n            avg = {k: tau_eff * sum(float(w / n) * a[k] for w, n, a in zip(weights, counts, deltas)) for k in state}\n        else:\n            avg = {k: sum(float(w) * a[k] for w, a in zip(weights, deltas)) for k in state}\n        model.load_state_dict({k: state[k] + avg[k] for k in state})\n\n        if rd % check == 0 or rd == rounds:\n            v = pauc(y[va], score_model(model, xx, va), 0.05)\n            if v > best + 1e-5:\n                best = v\n                best_state = {k: val.detach().cpu().clone() for k, val in model.state_dict().items()}\n                wait = 0\n            else:\n                wait += 1\n        if rd >= min_round and wait >= patience:\n            break\n\n    model.load_state_dict(best_state)\n    return (lambda ids: score_model(model, xx, ids)), None, dict(seconds=time.perf_counter() - t0)\n\n\ndef eval_scores(cs, cy, ts, ty, seed_tag):\n    alpha = 0.01\n    t_emp = emp_cut(cs[cy == 0], alpha)\n    t_oracle = emp_cut(ts[ty == 0], alpha)\n    locked_fpr, locked_rec, _, _ = eval_op(ty, ts, t_emp)\n    _, oracle_rec, _, _ = eval_op(ty, ts, t_oracle)\n\n    neg_s, pos_s = ts[ty == 0], ts[ty == 1]\n    rng = make_rng("day3_ut", seed_tag)\n    n_cal = min(2000, len(neg_s) // 2)\n    n_pos_cal = max(1, int(round(n_cal * 0.01)))\n    n_neg_cal = n_cal - n_pos_cal\n    uq, ur = [], []\n    for _ in range(5):\n        cn = rng.choice(len(neg_s), size=n_neg_cal, replace=False)\n        cp = rng.choice(len(pos_s), size=min(n_pos_cal, len(pos_s) // 2), replace=False)\n        en = neg_s[~np.isin(np.arange(len(neg_s)), cn)]\n        ep = pos_s[~np.isin(np.arange(len(pos_s)), cp)]\n        u = np.concatenate([neg_s[cn], pos_s[cp]])\n        tq = emp_cut(u, alpha)\n        tr = wilks_cut(u, alpha, delta=0.05)\n        uq.append(((en >= tq).mean(), (ep >= tq).mean(), float((en >= tq).mean() <= alpha)))\n        ur.append(((en >= tr).mean(), (ep >= tr).mean(), float((en >= tr).mean() <= alpha)))\n    uq, ur = np.array(uq), np.array(ur)\n\n    return dict(\n        auroc=pauc(ty, ts),\n        pauc01=pauc(ty, ts, 0.01),\n        pauc05=pauc(ty, ts, 0.05),\n        ap=float(average_precision_score(ty, ts)),\n        locked_fpr=locked_fpr,\n        locked_recall=locked_rec,\n        target_quantile_recall=oracle_rec,\n        ut_quantile_fpr=float(uq[:, 0].mean()),\n        ut_quantile_recall=float(uq[:, 1].mean()),\n        ut_quantile_cov=float(uq[:, 2].mean()),\n        ut_rat_fpr=float(ur[:, 0].mean()),\n        ut_rat_recall=float(ur[:, 1].mean()),\n        ut_rat_cov=float(ur[:, 2].mean()),\n    )\n\n\ndef run_temporal(z: zipfile.ZipFile, work: Path, opts: dict, seeds):\n    print("\\n================ PART 1: CHRONOLOGICAL TEMPORAL HOLDOUT (NF-UNSW-NB15-v3) ================", flush=True)\n    tc = np.load(io.BytesIO(z.read("temporal_cache.npz")), allow_pickle=False)\n    X, y, d, blk = tc["x"].astype(np.float32), tc["y"].astype(np.int8), tc["d"], tc["block"]\n\n    fit = np.flatnonzero(np.isin(blk, [0, 1]))\n    stop = np.flatnonzero(blk == 2)\n    cal = np.flatnonzero(blk == 3)\n    test = np.flatnonzero(blk == 4)\n    # Inner chronological validation for candidate selection: Block 0 -> Block 1\n    in_tr = np.flatnonzero(blk == 0)\n    in_va = np.flatnonzero(blk == 1)\n\n    t_dir = work / "temporal"\n    t_dir.mkdir(parents=True, exist_ok=True)\n    rows = []\n    tta_rows = []\n\n    for s in seeds:\n        clients_in = dirichlet_clients(in_tr, y, make_seed(s, "in"), K=3, alpha=0.5)\n        clients_full = dirichlet_clients(fit, y, make_seed(s, "full"), K=3, alpha=0.5)\n        for method in TEMPORAL_METHODS:\n            key = f"{method}_s{s}"\n            meta = t_dir / f"{key}.json"\n            if meta.exists():\n                rec = json.loads(meta.read_text())\n                rows.append(rec["metrics"])\n                if rec.get("tta_pair"):\n                    tta_rows.append(rec["tta_pair"])\n                continue\n\n            cands = candidate_grid(method)[: opts["trials"]]\n            scores_c = []\n            for j, cfg in enumerate(cands):\n                pred_c, _, _ = fit_method(X, y, d, in_tr, in_va, method, cfg, make_seed(s, "tmp_in", j), opts, clients_override=clients_in)\n                scores_c.append(pauc(y[stop], pred_c(stop), 0.05))\n            chosen = cands[int(np.argmax(scores_c))]\n\n            pred_src, pred_tta, info = fit_method(X, y, d, fit, stop, method, chosen, make_seed(s, "tmp_fin"), opts, clients_override=clients_full)\n            cs = pred_src(cal)\n            tta_pair = None\n            if method == "FedBN_TTA":\n                ts_pre = pred_src(test)\n                rng_tta = make_rng("tmp_tta", s)\n                neg_t, pos_t = test[y[test] == 0], test[y[test] == 1]\n                adapt_ids = np.sort(np.concatenate([\n                    rng_tta.choice(neg_t, size=min(1980, len(neg_t) // 2), replace=False),\n                    rng_tta.choice(pos_t, size=min(20, len(pos_t) // 2), replace=False),\n                ]))\n                ev_mask = ~np.isin(test, adapt_ids)\n                ts_post = pred_tta(test, adapt_ids=adapt_ids)\n                m_post = eval_scores(cs, y[cal], ts_post[ev_mask], y[test][ev_mask], f"tmp_{method}_{s}")\n                m_pre = eval_scores(cs, y[cal], ts_pre[ev_mask], y[test][ev_mask], f"tmp_{method}_pre_{s}")\n                tta_pair = dict(\n                    seed=s,\n                    pre_auroc=m_pre["auroc"], post_auroc=m_post["auroc"],\n                    pre_locked_fpr=m_pre["locked_fpr"], post_locked_fpr=m_post["locked_fpr"],\n                    pre_oracle_recall=m_pre["target_quantile_recall"], post_oracle_recall=m_post["target_quantile_recall"],\n                    delta_oracle_recall=m_post["target_quantile_recall"] - m_pre["target_quantile_recall"],\n                )\n                metrics = dict(method=method, seed=s, **m_post)\n            else:\n                ts = pred_src(test)\n                metrics = dict(method=method, seed=s, **eval_scores(cs, y[cal], ts, y[test], f"tmp_{method}_{s}"))\n\n            save_json(meta, dict(config=chosen, info=info, metrics=metrics, tta_pair=tta_pair))\n            rows.append(metrics)\n            if tta_pair:\n                tta_rows.append(tta_pair)\n            print(f"TEMPORAL DONE | seed={s} | {method:18s} | AUROC={metrics[\'auroc\']:.3f} | LockFPR={metrics[\'locked_fpr\']*100:5.2f}% | TgtQRec={metrics[\'target_quantile_recall\']*100:5.2f}% | UTqFPR={metrics[\'ut_quantile_fpr\']*100:4.2f}% | {info[\'seconds\']:.1f}s", flush=True)\n\n    df_t = pd.DataFrame(rows)\n    df_t.to_csv(work / "day3_temporal_runs.csv", index=False)\n    df_tm = df_t.groupby("method").mean(numeric_only=True).reset_index()\n    df_tm.to_csv(work / "day3_temporal_means.csv", index=False)\n    if tta_rows:\n        pd.DataFrame(tta_rows).to_csv(work / "day3_temporal_fedbn_tta.csv", index=False)\n    print("\\n=== TEMPORAL HOLDOUT MEANS (Block 4 Future Test) ===")\n    disp = df_tm[["method", "auroc", "pauc01", "locked_fpr", "locked_recall", "target_quantile_recall", "ut_quantile_fpr", "ut_quantile_recall", "ut_rat_cov"]].copy()\n    for c in disp.columns[3:]:\n        disp[c] *= 100.0\n    print(disp.round(2).to_string(index=False))\n\n\ndef run_ablation(z: zipfile.ZipFile, work: Path, opts: dict, domains, seeds):\n    print("\\n================ PART 2: MATCHED RETUNED FEATURE ABLATION (39 & 37 FEATURES) ================", flush=True)\n    cache = np.load(io.BytesIO(z.read("nf_cache.npz")), allow_pickle=False)\n    X_full, y, d = cache["x"].astype(np.float32), cache["y"].astype(np.int8), cache["d"]\n\n    a_dir = work / "ablation"\n    a_dir.mkdir(parents=True, exist_ok=True)\n    rows = []\n\n    for vname, drop_cols in ABLATION_VARIANTS.items():\n        keep_idx = [i for i, c in enumerate(FEATURES) if c not in drop_cols]\n        Xv = X_full[:, keep_idx]\n        for held in domains:\n            for s in seeds:\n                fit, stop, cal, _, test = split_outer_nf(y, d, [held], s)\n                src_domains = sorted(set(d[fit]))\n                inner = [(h, fit[d[fit] != h], stop[d[stop] != h], np.r_[fit[d[fit] == h], stop[d[stop] == h]]) for h in src_domains]\n                for method in ABLATION_METHODS:\n                    key = f"{vname}_{held}_s{s}_{method}"\n                    meta = a_dir / f"{key}.json"\n                    if meta.exists():\n                        rows.append(json.loads(meta.read_text())["metrics"])\n                        continue\n\n                    cands = candidate_grid(method)[: opts["trials"]]\n                    cand_scores = []\n                    for j, cfg in enumerate(cands):\n                        in_p = []\n                        for h, a_i, v_i, b_i in inner:\n                            pred_c, _, _ = fit_method(Xv, y, d, a_i, v_i, method, cfg, make_seed(s, h, 0), opts)\n                            in_p.append(pauc(y[b_i], pred_c(b_i), 0.05))\n                        cand_scores.append(float(np.mean(in_p)))\n                    chosen = cands[int(np.argmax(cand_scores))]\n\n                    pred_src, _, info = fit_method(Xv, y, d, fit, stop, method, chosen, make_seed(s, "final"), opts)\n                    cs = pred_src(cal)\n                    ts = pred_src(test)\n                    m = eval_scores(cs, y[cal], ts, y[test], key)\n                    metrics = dict(variant=vname, n_features=len(keep_idx), held=held, seed=s, method=method, **m)\n                    save_json(meta, dict(config=chosen, info=info, metrics=metrics))\n                    rows.append(metrics)\n                    print(f"ABLATION {vname:12s} | {held:18s} | s={s:4d} | {method:18s} | AUROC={m[\'auroc\']:.3f} | LockFPR={m[\'locked_fpr\']*100:5.1f}% | TgtQRec={m[\'target_quantile_recall\']*100:5.1f}%", flush=True)\n\n    df_a = pd.DataFrame(rows)\n    df_a.to_csv(work / "day3_ablation_runs.csv", index=False)\n    dom_m = df_a.groupby(["variant", "method", "held"]).mean(numeric_only=True).reset_index()\n    meth_m = dom_m.groupby(["variant", "method"]).mean(numeric_only=True).reset_index()\n    meth_m.to_csv(work / "day3_ablation_means.csv", index=False)\n    print("\\n=== FEATURE ABLATION MEANS (Domain-Averaged, % for rates) ===")\n    disp = meth_m[["variant", "method", "auroc", "pauc01", "locked_fpr", "target_quantile_recall", "ut_quantile_fpr", "ut_quantile_recall"]].copy()\n    for c in disp.columns[4:]:\n        disp[c] *= 100.0\n    print(disp.round(2).to_string(index=False))\n\n\ndef run_all(input_zip: Path, work: Path, run_part1: bool = True, run_part2: bool = True, smoke: bool = False):\n    work.mkdir(parents=True, exist_ok=True)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    opts = dict(\n        device=device,\n        smoke=smoke,\n        rounds=10 if smoke else 150,\n        min_round=5 if smoke else 30,\n        local_steps=2 if smoke else 8,\n        batch=256,\n        check_every=5,\n        patience=3 if smoke else 10,\n        trials=2 if smoke else 4,\n    )\n    seeds = [0] if smoke else SEEDS\n    domains = ["NF-BoT-IoT"] if smoke else DOMAINS\n    print(f"Running FLIDS Day 3 on device={device} | smoke={smoke}", flush=True)\n\n    with zipfile.ZipFile(input_zip) as z:\n        if run_part1:\n            run_temporal(z, work, opts, seeds)\n        if run_part2:\n            run_ablation(z, work, opts, domains, seeds)\n\n    out_zip = work / "FLIDS_Day3_Temporal_Ablation_Results.zip"\n    with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:\n        for p in sorted(work.glob("*.csv")):\n            zf.write(p, arcname=p.name)\n    print(f"\\nExported summary archive: {out_zip} ({out_zip.stat().st_size / 1024:.1f} KB)")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--input", required=True, type=Path)\n    ap.add_argument("--work", default=Path("./day3_work"), type=Path)\n    ap.add_argument("--smoke", action="store_true")\n    args = ap.parse_args()\n    run_all(args.input, args.work, smoke=args.smoke)\n\n\nif __name__ == "__main__":\n    main()\n'
Path('/kaggle/working/day3_temporal_ablation.py').write_text(CODE, encoding='utf-8')
print('Wrote /kaggle/working/day3_temporal_ablation.py', len(CODE), 'bytes')


In [ ]:
# Locate FLIDS_Master_V3_Fresh_Results.zip and run Day 3
import sys
from pathlib import Path
sys.path.insert(0, '/kaggle/working')
import day3_temporal_ablation as d3

hits = list(Path('/kaggle/input').rglob('FLIDS_Master_V3_Fresh_Results.zip'))
if not hits:
    raise FileNotFoundError('Attach fragoa/notebook6ceac598dd so FLIDS_Master_V3_Fresh_Results.zip is in /kaggle/input')
INPUT_ZIP = hits[0]
print('Using input:', INPUT_ZIP)

d3.run_all(
    input_zip=INPUT_ZIP,
    work=Path(WORK_DIR),
    run_part1=RUN_TEMPORAL,
    run_part2=RUN_ABLATION,
    smoke=SMOKE_TEST,
)


In [ ]:
# Display summary tables
import pandas as pd
from pathlib import Path
w = Path(WORK_DIR)
if (w / 'day3_temporal_means.csv').exists():
    print('=== Temporal Holdout Means ===')
    display(pd.read_csv(w / 'day3_temporal_means.csv'))
if (w / 'day3_ablation_means.csv').exists():
    print('=== Feature Ablation Means ===')
    display(pd.read_csv(w / 'day3_ablation_means.csv'))
